# Session 10 — Scenario Comparison & Variance Reduction Methods

**Course:** Decision Modelling — Simulation Part
**Session:** 10 of 12 (simulation portion)

## Learning objectives

By the end of this session you should be able to:

- Choose an appropriate experimental design for comparing simulated scenarios
- Apply variance-reduction methods (Common Random Numbers) to DES comparisons
- Perform statistically rigorous comparisons across two or more scenarios (paired tests, ANOVA)

## Format note: flipped classroom

- **Pre-class (screencast + this notebook's Part A):** build a new SimPy model — a production stage with a **quality-control rework loop** (conditional branching: an item may be sent back for reprocessing). Watch the screencast and run Part A yourself before class.
- **In-class (Part B):** the statistical core — CRN for DES (which turns out to be genuinely trickier than the Monte Carlo case in Session 4), and comparing more than two scenarios with ANOVA.

## Where this session fits

- **Looking back:** Session 4 introduced CRN for Monte Carlo — reusing one shared random stream across two policies. Session 9 gave us tools for rigorous single-scenario steady-state estimates. Today we combine both ideas: rigorous **comparison** between DES scenarios.
- **New SimPy feature today:** **conditional branching / a rework loop** — after processing, an item may probabilistically be sent back through the same process again, rather than always exiting.
- **Looking ahead:** Session 11 combines everything from Sessions 6–10 into one capstone production/inventory model.


## Further reading (supplementary, not required)

- A paper or chapter on variance-reduction techniques in simulation (CRN specifically) and on experimental design for comparing simulated alternatives (paired tests, ANOVA) — check the course reading list for the current reference.


---
# Part A (pre-class): a production stage with a rework loop

Watch the accompanying screencast for a walkthrough of the code below, then run it yourself before class.

## A new SimPy mechanic: conditional branching

So far, every entity in our models has moved through the system in one direction: arrive, get served, leave. Many real processes aren't this simple — a quality-control check might send a fraction of items **back** for rework before they can exit. This is **conditional branching**: after finishing a step, we sample a random outcome and decide, based on that outcome, whether the entity's journey continues normally or loops back.


In [1]:
import simpy
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
from itertools import combinations

def workpiece(env, wid, server, service_time_fn, rework_prob, rng_service, rng_rework, records):
    arrival_time = env.now
    total_wait = 0.0
    passes = 0

    while True:
        request_time = env.now
        with server.request() as req:
            yield req
            total_wait += env.now - request_time
            service_time = service_time_fn(rng_service)
            yield env.timeout(service_time)
        passes += 1

        if rng_rework.random() >= rework_prob:
            break   # passed QC -- exits the system
        # else: loop back to the top of the while loop for another pass (rework)

    departure_time = env.now
    records.append({
        'id': wid, 'arrival_time': arrival_time, 'total_wait': total_wait,
        'passes': passes, 'time_in_system': departure_time - arrival_time,
    })

def workpiece_generator(env, mean_interarrival, server, service_time_fn, rework_prob,
                          rng_arrival, rng_service, rng_rework, records):
    wid = 0
    while True:
        yield env.timeout(rng_arrival.exponential(mean_interarrival))
        wid += 1
        env.process(workpiece(env, wid, server, service_time_fn, rework_prob,
                                rng_service, rng_rework, records))


Notice we're using **three separate random-number generators** — one each for interarrival times, service times, and rework decisions — instead of one shared generator for everything. This is deliberate, and it's the setup Part B needs: keeping each *source* of randomness in its own stream is what makes it possible to reuse specific streams across scenario comparisons later.


In [2]:
mean_interarrival = 5.0
mean_service = 3.0
rework_prob = 0.20
run_length = 3000.0

rng_arrival = np.random.default_rng(seed=1)
rng_service = np.random.default_rng(seed=2)
rng_rework = np.random.default_rng(seed=3)

exponential_service = lambda rng: rng.exponential(mean_service)

env = simpy.Environment()
server = simpy.Resource(env, capacity=1)
records = []

env.process(workpiece_generator(env, mean_interarrival, server, exponential_service, rework_prob,
                                  rng_arrival, rng_service, rng_rework, records))
env.run(until=run_length)

df_a = pd.DataFrame(records)
print(f"Items completed: {len(df_a)}")
print(f"Mean number of passes through the server: {df_a['passes'].mean():.3f}")
print(f"Mean time in system: {df_a['time_in_system'].mean():.2f}")
print(f"Distribution of passes:\n{df_a['passes'].value_counts().sort_index()}")


Items completed: 609
Mean number of passes through the server: 1.271
Mean time in system: 12.66
Distribution of passes:
passes
1    476
2    109
3     16
4      8
Name: count, dtype: int64


Notice most items exit after 1 pass, some need 2, fewer need 3, and so on — a geometric-like pattern, exactly as you'd expect from repeated independent QC checks with a fixed rework probability. Part A is done here — the rest of this notebook (Part B) is for class.


---
# Part B (in-class): comparing scenarios rigorously

## 1. The decision question

Management is considering a quality-improvement investment that would reduce the rework probability from **20% to 5%**. They want to know: how much does this actually improve mean time in system, and is the improvement large enough to be confident it's real (not just simulation noise)?

## 2. CRN for DES: trickier than the Monte Carlo case

In Session 4, applying CRN was simple: reuse the *same* random draws for both policies, since both policies consumed exactly the same *number* of random draws per replication (one demand draw per day, for both).

**DES comparisons are not so simple.** Here, an item that gets reworked consumes an *extra* service-time draw that a same item, under a different (lower) rework probability, might not need. Once one item's rework outcome differs between the two scenarios, every job *after* it in the queue will draw from a different position in the shared service-time stream — the two scenarios' randomness silently **desynchronizes** downstream. This is a genuine, well-known challenge in applying CRN to DES models with random branching, not just an implementation detail to brush past.

**A practical best-effort approach:** keep separate random streams for each *source* of randomness (arrivals, service times, rework decisions) and reuse the interarrival and service streams identically across scenarios. This maximizes shared randomness for as long as the two scenarios' event sequences stay aligned, even though perfect synchronization can't be guaranteed once branching outcomes diverge. Let's test empirically whether this still helps, compared to using entirely independent randomness for each scenario.


In [3]:
def run_rework_system(mean_interarrival, mean_service, rework_prob, run_length,
                        seed_arrival, seed_service, seed_rework):
    rng_arrival = np.random.default_rng(seed_arrival)
    rng_service = np.random.default_rng(seed_service)
    rng_rework = np.random.default_rng(seed_rework)

    env = simpy.Environment()
    server = simpy.Resource(env, capacity=1)
    records = []

    service_fn = lambda rng: rng.exponential(mean_service)
    env.process(workpiece_generator(env, mean_interarrival, server, service_fn, rework_prob,
                                      rng_arrival, rng_service, rng_rework, records))
    env.run(until=run_length)
    return pd.DataFrame(records)

def replication_mean_time(mean_interarrival, mean_service, rework_prob, run_length,
                            seed_arrival, seed_service, seed_rework):
    df = run_rework_system(mean_interarrival, mean_service, rework_prob, run_length,
                             seed_arrival, seed_service, seed_rework)
    return df['time_in_system'].mean()

n_reps = 30
run_length = 3000.0

# --- Independent streams: every scenario x replication gets entirely fresh seeds ---
baseline_indep = []
improved_indep = []
for i in range(n_reps):
    baseline_indep.append(replication_mean_time(mean_interarrival, mean_service, 0.20, run_length,
                                                  seed_arrival=1000 + i, seed_service=2000 + i, seed_rework=3000 + i))
    improved_indep.append(replication_mean_time(mean_interarrival, mean_service, 0.05, run_length,
                                                  seed_arrival=5000 + i, seed_service=6000 + i, seed_rework=7000 + i))

baseline_indep = np.array(baseline_indep)
improved_indep = np.array(improved_indep)

# --- CRN: SAME arrival and service seeds across scenarios; only rework stream differs by construction ---
baseline_crn = []
improved_crn = []
for i in range(n_reps):
    baseline_crn.append(replication_mean_time(mean_interarrival, mean_service, 0.20, run_length,
                                                seed_arrival=1000 + i, seed_service=2000 + i, seed_rework=3000 + i))
    improved_crn.append(replication_mean_time(mean_interarrival, mean_service, 0.05, run_length,
                                                seed_arrival=1000 + i, seed_service=2000 + i, seed_rework=3000 + i))

baseline_crn = np.array(baseline_crn)
improved_crn = np.array(improved_crn)


In [4]:
def paired_ci(a, b, confidence=0.95):
    diff = a - b
    n = len(diff)
    mean_diff = diff.mean()
    sem = diff.std(ddof=1) / np.sqrt(n)
    half_width = stats.t.ppf((1 + confidence) / 2, df=n - 1) * sem
    return mean_diff, half_width

diff_indep, hw_indep = paired_ci(baseline_indep, improved_indep)
diff_crn, hw_crn = paired_ci(baseline_crn, improved_crn)

print("Independent streams:")
print(f"  Mean improvement (baseline - improved): {diff_indep:.3f}")
print(f"  95% CI half-width: {hw_indep:.3f}")

print("\nShared arrival/service streams (CRN-style):")
print(f"  Mean improvement (baseline - improved): {diff_crn:.3f}")
print(f"  95% CI half-width: {hw_crn:.3f}")

print(f"\nVariance reduction factor: {hw_indep / hw_crn:.2f}x narrower CI with shared streams")


Independent streams:
  Mean improvement (baseline - improved): 6.604
  95% CI half-width: 1.494

Shared arrival/service streams (CRN-style):
  Mean improvement (baseline - improved): 6.654
  95% CI half-width: 1.062

Variance reduction factor: 1.41x narrower CI with shared streams


### Discussion: did CRN still help, even though perfect synchronization isn't possible?

Even partial, imperfect synchronization (shared arrival and service streams, up until branching outcomes diverge) can still meaningfully reduce variance, especially early in a run before much desynchronization has accumulated. This is a useful, realistic takeaway: **CRN in DES is a "best effort, often still worthwhile" technique, not an all-or-nothing guarantee** the way it was for the Monte Carlo Newsvendor model in Session 4.


## 3. Comparing more than two scenarios: ANOVA

Suppose instead we want to compare **three** rework-probability scenarios at once: `0.05`, `0.15`, `0.25` (perhaps representing "no investment," "partial investment," and "current process," in some order management wants evaluated together). Running pairwise t-tests for every pair gets statistically messy as the number of scenarios grows — this is exactly the situation **ANOVA** (Analysis of Variance) is designed for: testing whether *any* of several group means differ, in one combined test.


In [5]:
rework_probs = [0.05, 0.15, 0.25]
n_reps = 30
run_length = 3000.0

scenario_results = {}
for scenario_idx, rp in enumerate(rework_probs):
    means = []
    for i in range(n_reps):
        # NOTE: seed base is offset by scenario_idx*10000, so each scenario gets its own
        # independent set of random streams -- no sharing across scenarios here.
        means.append(replication_mean_time(mean_interarrival, mean_service, rp, run_length,
                                             seed_arrival=10000 + scenario_idx * 10000 + i,
                                             seed_service=20000 + scenario_idx * 10000 + i,
                                             seed_rework=30000 + scenario_idx * 10000 + i))
    scenario_results[rp] = np.array(means)

summary = pd.DataFrame({
    'rework_prob': rework_probs,
    'mean_time_in_system': [scenario_results[rp].mean() for rp in rework_probs],
    'std': [scenario_results[rp].std(ddof=1) for rp in rework_probs],
})
print(summary.round(3))

f_stat, p_value = stats.f_oneway(*[scenario_results[rp] for rp in rework_probs])
print(f"\nOne-way ANOVA: F = {f_stat:.3f}, p-value = {p_value:.6f}")


   rework_prob  mean_time_in_system    std
0         0.05                8.774  1.508
1         0.15               11.470  2.558
2         0.25               18.078  3.825

One-way ANOVA: F = 87.951, p-value = 0.000000


### Interpreting the ANOVA result

A small p-value tells us **at least one** scenario's mean differs from the others — but not *which* one(s). If the overall F-test is significant, the standard follow-up is **pairwise comparisons**, using a correction for the fact that we're now running multiple tests (to avoid inflating the false-positive rate). A simple, widely applicable correction is the **Bonferroni correction**: divide your significance threshold by the number of pairwise comparisons you're running.


In [6]:
pairs = list(combinations(rework_probs, 2))
alpha = 0.05
bonferroni_alpha = alpha / len(pairs)

print(f"Number of pairwise comparisons: {len(pairs)}")
print(f"Bonferroni-adjusted significance threshold: {bonferroni_alpha:.4f}\n")

for rp1, rp2 in pairs:
    t_stat, p_val = stats.ttest_ind(scenario_results[rp1], scenario_results[rp2])
    significant = "significant" if p_val < bonferroni_alpha else "not significant"
    print(f"rework_prob {rp1} vs {rp2}: t = {t_stat:.3f}, p = {p_val:.6f}  ({significant} at Bonferroni-adjusted alpha)")


Number of pairwise comparisons: 3
Bonferroni-adjusted significance threshold: 0.0167

rework_prob 0.05 vs 0.15: t = -4.972, p = 0.000006  (significant at Bonferroni-adjusted alpha)
rework_prob 0.05 vs 0.25: t = -12.394, p = 0.000000  (significant at Bonferroni-adjusted alpha)
rework_prob 0.15 vs 0.25: t = -7.866, p = 0.000000  (significant at Bonferroni-adjusted alpha)


## Exercise (guided): does CRN help the three-scenario comparison too?

Repeat the three-scenario ANOVA comparison above, but this time use the **same** `seed_arrival` and `seed_service` values across all three `rework_prob` scenarios (only vary `seed_rework` by replication, matching the CRN-style approach from Section 2).

Your task:

1. Re-run the three-scenario comparison with shared arrival/service seeds across scenarios (vary only the replication index `i`, not the scenario)
2. Recompute the one-way ANOVA F-statistic and p-value
3. Recompute the pairwise comparisons with Bonferroni correction
4. Compare the standard deviations within each scenario to the independent-streams version above — did CRN-style synchronization reduce within-scenario variance here too?


In [7]:
# EXERCISE
# TODO: 1. Re-run the three rework_prob scenarios using shared seed_arrival/seed_service
#          across scenarios (same seeds as scenario_results used, but keep rework_prob varying)

# TODO: 2. Recompute stats.f_oneway on the new results

# TODO: 3. Recompute pairwise t-tests with Bonferroni correction

# TODO: 4. Compare standard deviations to the independent-streams version (`summary` above)


<details>
<summary>Solution (click to expand)</summary>

```python
# SOLUTION
scenario_results_crn = {}
for rp in rework_probs:
    means = []
    for i in range(n_reps):
        means.append(replication_mean_time(mean_interarrival, mean_service, rp, run_length,
                                             seed_arrival=10000 + i, seed_service=20000 + i, seed_rework=30000 + i))
        # Note: seed_arrival and seed_service are identical to the independent version above
        # because we already used the SAME (10000+i, 20000+i) pattern for every rp -- the only
        # thing that varies by scenario here is rework_prob itself, and seed_rework is also
        # shared across scenarios (30000+i) by construction, since it was written that way above.
    scenario_results_crn[rp] = np.array(means)

summary_crn = pd.DataFrame({
    'rework_prob': rework_probs,
    'mean_time_in_system': [scenario_results_crn[rp].mean() for rp in rework_probs],
    'std': [scenario_results_crn[rp].std(ddof=1) for rp in rework_probs],
})
print(summary_crn.round(3))

f_stat_crn, p_value_crn = stats.f_oneway(*[scenario_results_crn[rp] for rp in rework_probs])
print(f"\nOne-way ANOVA (CRN-style): F = {f_stat_crn:.3f}, p-value = {p_value_crn:.6f}")

for rp1, rp2 in pairs:
    t_stat, p_val = stats.ttest_ind(scenario_results_crn[rp1], scenario_results_crn[rp2])
    significant = "significant" if p_val < bonferroni_alpha else "not significant"
    print(f"rework_prob {rp1} vs {rp2}: t = {t_stat:.3f}, p = {p_val:.6f}  ({significant})")
```

</details>


### Discussion questions

1. In Section 3, `stats.ttest_ind` (independent two-sample t-test) was used for pairwise comparisons — is that the right test if the underlying replications shared arrival/service seeds across scenarios? What test would be appropriate for paired (CRN-style) data instead?
2. Why does Bonferroni correction make it *harder* to declare a pairwise difference significant as the number of scenarios grows? What would you do if you needed to compare, say, 10 scenarios pairwise?
3. Looking back at Section 2's discussion: would you expect CRN's benefit to be larger or smaller when comparing three fairly different rework probabilities (0.05 vs 0.25) versus two close ones? Why?


## Wrap-up

Today (Part A, pre-class) we:

- Introduced conditional branching via a QC rework loop, using separate random-number streams per source of randomness

Today (Part B, in-class) we:

- Confronted the real complication in applying CRN to DES: random branching can desynchronize shared streams over time, unlike the clean Monte Carlo case in Session 4
- Empirically tested whether CRN-style stream-sharing still reduces variance despite this limitation (it did, meaningfully)
- Extended scenario comparison beyond two alternatives using one-way ANOVA, followed by Bonferroni-corrected pairwise comparisons

**Next session:** Advanced DES Practice — we bring together every SimPy feature from Sessions 6–10 (resources, pooling, multi-stage flow, interrupts, branching) plus `simpy.Container` and finite buffers, to build a full inventory or production-line capstone model.
